## Phase 5: Modelling & Evaluation

In [ ]:
import pandas as pd
import joblib
import os
from google.colab import drive
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

load_dir = '/content/drive/MyDrive/processed_data'
X_train = pd.read_csv(f'{load_dir}/X_train.csv')
X_test = pd.read_csv(f'{load_dir}/X_test.csv')
y_train = pd.read_csv(f'{load_dir}/y_train.csv').values.ravel()
y_test = pd.read_csv(f'{load_dir}/y_test.csv').values.ravel()

# Encoding Target Variable
target_le = LabelEncoder()
y_train_num = target_le.fit_transform(y_train)
y_test_num = target_le.transform(y_test)

# 2. Baseline: Dummy Classifier
dummy_clf = DummyClassifier(strategy='most_frequent')
dummy_clf.fit(X_train, y_train_num)
y_pred_dummy = dummy_clf.predict(X_test)

print("--- Dummy Classifier Baseline ---")
print(f"Weighted F1 Score: {f1_score(y_test_num, y_pred_dummy, average='weighted'):.4f}\n")

# 3. Random Forest
rf_clf = RandomForestClassifier(class_weight='balanced', random_state=42)
rf_clf.fit(X_train, y_train_num)
y_pred_rf = rf_clf.predict(X_test)

print("--- Random Forest Classifier ---")
print(f"Weighted F1 Score: {f1_score(y_test_num, y_pred_rf, average='weighted'):.4f}\n")

# 4. XGBoost with Hyperparameter Tuning & 5-Fold Stratified CV
xgb_base = XGBClassifier(random_state=42, eval_metric='mlogloss')
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid_search = GridSearchCV(xgb_base, param_grid, scoring='f1_weighted', cv=cv, n_jobs=-1)
grid_search.fit(X_train, y_train_num)

best_xgb = grid_search.best_estimator_
y_pred_xgb = best_xgb.predict(X_test)

print("--- Optimized XGBoost Classifier ---")
print(f"Best Hyperparameters: {grid_search.best_params_}")
print(f"Weighted F1 Score: {f1_score(y_test_num, y_pred_xgb, average='weighted'):.4f}\n")

print("Classification Report (XGBoost):")
print(classification_report(y_test_num, y_pred_xgb, target_names=target_le.classes_.astype(str)))

save_dir = '/content/drive/MyDrive/models'
os.makedirs(save_dir, exist_ok=True)
joblib.dump(best_xgb, f'{save_dir}/xgboost_model.pkl')
joblib.dump(target_le, f'{save_dir}/target_encoder.pkl')
print(f"\nBest XGBoost model and target encoder saved successfully to {save_dir}.")

--- Dummy Classifier Baseline ---
Weighted F1 Score: 0.9408

--- Random Forest Classifier ---
Weighted F1 Score: 0.9418

--- Optimized XGBoost Classifier ---
Best Hyperparameters: {'learning_rate': 0.1, 'max_depth': 7, 'n_estimators': 100}
Weighted F1 Score: 0.9426

Classification Report (XGBoost):
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        10
           1       0.33      0.03      0.06        95
           2       0.96      1.00      0.98      2536

    accuracy                           0.96      2641
   macro avg       0.43      0.34      0.35      2641
weighted avg       0.94      0.96      0.94      2641


Best XGBoost model and target encoder saved successfully to /content/drive/MyDrive/models.
